In [11]:
import torch
import platform

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(
            f"GPU {i}: {torch.cuda.get_device_name(i)} | "
            f"VRAM: {props.total_memory / 1024**3:.2f} GB"
        )

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA available: True
GPU count: 2
GPU 0: Tesla T4 | VRAM: 14.56 GB
GPU 1: Tesla T4 | VRAM: 14.56 GB


In [12]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print("CUDA_VISIBLE_DEVICES:", os.environ["CUDA_VISIBLE_DEVICES"])

CUDA_VISIBLE_DEVICES: 0


In [13]:
from pathlib import Path

print("Relevant uploaded files:\n")

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file() and (
        p.name == "train.jsonl"
        or "localsql-phase4" in str(p).lower()
    ):
        print(p)

Relevant uploaded files:

/kaggle/input/datasets/hassanch6138/localsql-phase4-input/train.jsonl
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/.gitignore
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/pyproject.toml
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/README.md
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/uv.lock
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/.python-version
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/CLAUDE.md
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/tests/__init__.py
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase4-src/localsql-phase3-src/PROJECT.md
/kaggle/input/datasets/hassanch6138/localsql-phase4-input/localsql-phase

In [14]:
from pathlib import Path
import shutil

SRC = Path(
    "/kaggle/input/datasets/hassanch6138/"
    "localsql-phase4-input/localsql-phase4-src"
)

TRAIN = Path(
    "/kaggle/input/datasets/hassanch6138/"
    "localsql-phase4-input/train.jsonl"
)

WORK = Path("/kaggle/working/localsql")

if WORK.exists():
    shutil.rmtree(WORK)

# Copy the current Phase 4 repo only.
# Explicitly ignore the accidentally nested old Phase 3 source.
shutil.copytree(
    SRC,
    WORK,
    ignore=shutil.ignore_patterns("localsql-phase3-src"),
)

(WORK / "data" / "processed").mkdir(parents=True, exist_ok=True)
shutil.copy2(TRAIN, WORK / "data" / "processed" / "train.jsonl")

print("Working repo:", WORK)
print("Training file:", WORK / "data/processed/train.jsonl")
print("Runner exists:", (WORK / "scripts/run_qlora_smoke.py").exists())
print("Train config exists:", (WORK / "configs/train.yaml").exists())
print("Nested Phase 3 copied:", (WORK / "localsql-phase3-src").exists())

Working repo: /kaggle/working/localsql
Training file: /kaggle/working/localsql/data/processed/train.jsonl
Runner exists: True
Train config exists: True
Nested Phase 3 copied: False


In [15]:
from pathlib import Path
import json

train_path = Path("/kaggle/working/localsql/data/processed/train.jsonl")

count = 0
db_ids = set()

with train_path.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            row = json.loads(line)
            count += 1
            if "db_id" in row:
                db_ids.add(row["db_id"])

print("Training examples:", count)
print("Unique db_ids:", len(db_ids))

Training examples: 6067
Unique db_ids: 62


In [16]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [17]:
!python -m pip install -q -e . "transformers>=4.51" accelerate bitsandbytes peft trl

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 43.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.8 MB/s eta 0:00:00
  Building editable for localsql (pyproject.toml) ... done


In [18]:
import torch
import transformers
import accelerate
import bitsandbytes
import peft
import trl

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
transformers: 5.0.0
accelerate: 1.13.0
bitsandbytes: 0.50.2
peft: 0.19.1
trl: 1.13.0
CUDA: 12.8
GPU: Tesla T4


In [19]:
!python scripts/run_qlora_smoke.py \
    --run-id qlora-smoke-1 \
    --token-profile

config.json: 100%|█████████████████████████████| 727/727 [00:00<00:00, 3.20MB/s]
tokenizer_config.json: 9.38kB [00:00, 29.7MB/s]
vocab.json: 2.78MB [00:00, 96.5MB/s]
merges.txt: 1.67MB [00:00, 123MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 20.9MB/s]
Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 6067 real training examples ...
{
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 6067,
  "prompt_only_token_stats": {
    "count": 6067,
    "min": 509.0,
    "median": 2468.0,
    "p90": 7956.4,
    "p95": 27890.0,
    "p99": 27918.34,
    "max": 27995.0
  },
  "prompt_only_count_above_warn": 1397,
  "prompt_only_count_above_hard": 539,
  "full_sft_sequence_token_stats": {
    "count": 6067,
    "min": 549.0,
    "median": 2506.0,
    "p90": 8013.8,
    "p95": 27927.0,
    "p99": 27975.0,
    "max": 28082.0
  },
  "full_sft_sequence_count_above_warn": 1398,
  "full_sft_sequence_count_above

In [20]:
import json
from pathlib import Path
from pprint import pprint

profile_path = Path(
    "/kaggle/working/localsql/"
    "data/runs/qlora-smoke-1/train_token_profile.json"
)

print("Exists:", profile_path.exists())

with profile_path.open("r", encoding="utf-8") as f:
    profile = json.load(f)

pprint(profile)

Exists: True
{'configured_max_seq_length': 4096,
 'example_count': 6067,
 'full_sft_sequence_count_above_hard': 539,
 'full_sft_sequence_count_above_warn': 1398,
 'full_sft_sequence_token_stats': {'count': 6067,
                                   'max': 28082.0,
                                   'median': 2506.0,
                                   'min': 549.0,
                                   'p90': 8013.8,
                                   'p95': 27927.0,
                                   'p99': 27975.0},
 'note': 'max_seq_length was NOT auto-adjusted from this profile -- review '
         'before changing configs/train.yaml.',
 'prompt_only_count_above_hard': 539,
 'prompt_only_count_above_warn': 1397,
 'prompt_only_token_stats': {'count': 6067,
                             'max': 27995.0,
                             'median': 2468.0,
                             'min': 509.0,
                             'p90': 7956.4,
                             'p95': 27890.0,
            

In [25]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [26]:
!python scripts/run_qlora_smoke.py \
    --run-id qlora-smoke-preflight \
    --max-train-examples 16 \
    --max-steps 1 \
    --skip-verify

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
model.safetensors.index.json: 32.8kB [00:00, 70.8MB/s]
Fetching 3 files: 100%|███████████████████████████| 3/3 [00:26<00:00,  8.89s/it]
Download complete: 100%|████████████████████| 8.04G/8.04G [00:26<00:00, 301MB/s]
Loading weights: 100%|█| 398/398 [00:02<00:00, 138.08it/s, Materializing param=m
generation_config.json: 100%|███████████████████| 238/238 [00:00<00:00, 964kB/s]
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 16 examples ...
  usable: 16  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=1) ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '1.523', 'grad_norm': '7.31', 'learning_rate': '0', 'epoch': '0.5'}    
{'train_runtime': '114', 'train_samples_per_second': '0.07', 'train_steps_per_second': '0.009', 'train_loss':

In [27]:
%cd /kaggle/working/localsql

/kaggle/working/localsql


In [29]:
!python scripts/run_qlora_smoke.py \
    --run-id qlora-smoke-200-profile \
    --token-profile \
    --max-train-examples 200

Tokenizer loaded (revision cdbee75f17c01a7cc42f958dc650907174af0554). Profiling 200 real training examples ...
{
  "tokenizer_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "example_count": 200,
  "prompt_only_token_stats": {
    "count": 200,
    "min": 1331.0,
    "median": 2704.0,
    "p90": 3491.2,
    "p95": 3500.05,
    "p99": 3524.06,
    "max": 3537.0
  },
  "prompt_only_count_above_warn": 0,
  "prompt_only_count_above_hard": 0,
  "full_sft_sequence_token_stats": {
    "count": 200,
    "min": 1355.0,
    "median": 2766.5,
    "p90": 3551.2,
    "p95": 3559.15,
    "p99": 3593.0,
    "max": 3593.0
  },
  "full_sft_sequence_count_above_warn": 0,
  "full_sft_sequence_count_above_hard": 0,
  "configured_max_seq_length": 4096,
  "note": "max_seq_length was NOT auto-adjusted from this profile -- review before changing configs/train.yaml."
}

Wrote /kaggle/working/localsql/data/runs/qlora-smoke-200-profile/train_token_profile.json


In [30]:
!PYTORCH_ALLOC_CONF=expandable_segments:True \
python scripts/run_qlora_smoke.py \
    --run-id qlora-smoke-1-alloc-retry \
    --max-train-examples 200 \
    --max-steps 20

Loading Qwen/Qwen3-4B-Instruct-2507 for QLoRA training (4-bit nf4) ...
Loading weights: 100%|█| 398/398 [00:02<00:00, 147.54it/s, Materializing param=m
Loaded. Resolved revision: cdbee75f17c01a7cc42f958dc650907174af0554  GPU: Tesla T4
Trainable params: 33,030,144 / 2,238,840,320
Building completion-only-masked SFT encodings for 200 examples ...
  usable: 200  skipped (exceeds max_seq_length=4096): 0
Training (max_steps=20) ...
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '1.501', 'grad_norm': '6.075', 'learning_rate': '0', 'epoch': '0.04'}  
{'loss': '1.423', 'grad_norm': '5.727', 'learning_rate': '0.0001', 'epoch': '0.08'}
{'loss': '0.671', 'grad_norm': '3.146', 'learning_rate': '9.474e-05', 'epoch': '0.12'}
{'loss': '0.7013', 'grad_norm': '2.237', 'learning_rate': '8.947e-05', 'epoch': '0.16'}
{'loss': '0.5271', 'grad_norm': '1.313', 'learning_rate': '8.421e-05', 'epoch': '0.2'}
{'loss': '0.3149', 'grad_norm': '1.563', 'learning_rate': 

In [31]:
from pathlib import Path
import json

run = Path(
    "/kaggle/working/localsql/data/runs/"
    "qlora-smoke-1-alloc-retry"
)

files = [
    "run_config.json",
    "summary.json",
    "adapter_verification.json",
]

for filename in files:
    path = run / filename
    print(f"\n{'=' * 20} {filename} {'=' * 20}")

    if path.exists():
        with path.open("r", encoding="utf-8") as f:
            print(json.dumps(json.load(f), indent=2))
    else:
        print("MISSING")


print("\n==================== TRAIN METRICS ====================")

metrics_path = run / "train_metrics.jsonl"

if metrics_path.exists():
    lines = metrics_path.read_text(encoding="utf-8").splitlines()
    print("Metric records:", len(lines))

    for line in lines:
        print(line)
else:
    print("MISSING")


print("\n==================== ORIGINAL 6067 PROFILE ====================")

profile_path = Path(
    "/kaggle/working/localsql/data/runs/"
    "qlora-smoke-1/train_token_profile.json"
)

if profile_path.exists():
    with profile_path.open("r", encoding="utf-8") as f:
        print(json.dumps(json.load(f), indent=2))
else:
    print("MISSING")


==================== run_config.json ====================
{
  "run_id": "qlora-smoke-1-alloc-retry",
  "model_id": "Qwen/Qwen3-4B-Instruct-2507",
  "resolved_revision": "cdbee75f17c01a7cc42f958dc650907174af0554",
  "quantization": {
    "load_in_4bit": true,
    "quant_type": "nf4",
    "double_quant": true,
    "compute_dtype": "float16"
  },
  "lora": {
    "r": 16,
    "alpha": 32,
    "dropout": 0.05,
    "target_modules": [
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj"
    ],
    "task_type": "CAUSAL_LM"
  },
  "optimization": {
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 8,
    "learning_rate": 0.0001,
    "warmup_ratio": 0.05,
    "gradient_checkpointing": true,
    "optim": "paged_adamw_8bit",
    "seed": 42
  },
  "max_seq_length": 4096,
  "completion_only_loss": true,
  "max_steps": 20,
  "requested_train_examples": 200,
  "train_file": "/kaggle/working/localsql/data/processe

In [33]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/localsql/data/runs")
out = Path("/kaggle/working/localsql-phase4-smoke-evidence.zip")

files_to_keep = [
    root / "qlora-smoke-1" / "train_token_profile.json",
    root / "qlora-smoke-200-profile" / "train_token_profile.json",

    root / "qlora-smoke-preflight" / "run_config.json",
    root / "qlora-smoke-preflight" / "summary.json",
    root / "qlora-smoke-preflight" / "train_metrics.jsonl",

    root / "qlora-smoke-1-alloc-retry" / "run_config.json",
    root / "qlora-smoke-1-alloc-retry" / "summary.json",
    root / "qlora-smoke-1-alloc-retry" / "train_metrics.jsonl",
]

with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as z:
    for p in files_to_keep:
        if p.exists():
            z.write(p, p.relative_to(root.parent))
            print("Added:", p)
        else:
            print("Missing/skipped:", p)

print("\nCreated:", out)
print("Size MB:", round(out.stat().st_size / 1024**2, 2))

Added: /kaggle/working/localsql/data/runs/qlora-smoke-1/train_token_profile.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-200-profile/train_token_profile.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-preflight/run_config.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-preflight/summary.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-preflight/train_metrics.jsonl
Added: /kaggle/working/localsql/data/runs/qlora-smoke-1-alloc-retry/run_config.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-1-alloc-retry/summary.json
Added: /kaggle/working/localsql/data/runs/qlora-smoke-1-alloc-retry/train_metrics.jsonl

Created: /kaggle/working/localsql-phase4-smoke-evidence.zip
Size MB: 0.01
